In [ ]:
from the_well.data.datasets import TrajectoryData, TrajectoryMetadata
from the_well.benchmark.models import UNetClassic, UNetConvNext
from the_well.data.normalization import ZScoreNormalization
from the_well.data import WellDataset

from the_well.benchmark.metrics.spatial import VRMSE
from typing import Sequence, Dict, Tuple, Optional
from einops import rearrange
from pathlib import Path
import fsspec
import pickle
import torch
import yaml
import sys
import os

#from unittest.mock import MagicMock
# Mock torch_harmonics as a proper package with all submodules neuralop touches
#mock_th = MagicMock()
#mock_th.__path__ = []          # marks it as a package
#mock_th.__spec__ = MagicMock() # satisfies importlib checks

#sys.modules['torch_harmonics']          = mock_th
#sys.modules['torch_harmonics.examples'] = MagicMock()
#sys.modules['torch_harmonics.layers']   = MagicMock()
#sys.modules['torch_harmonics.grids']    = MagicMock()

PROJECT_ROOT = "/Users/gabrieleaccarino/Workspace/waveletflux"
sys.path.insert(0, PROJECT_ROOT)

from waveletflux.utils.well_data import get_well_dataset, get_well_inputs
from waveletflux.modules.neural_ops import NeuralOperatorModule
from waveletflux.utils.rollout import rollout_normalized

device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")

/opt/anaconda3/envs/diffusion_env/lib/python3.12/site-packages/timm/models/layers/__init__.py:48: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)


# Load Data

In [2]:
well_base_path = f"{PROJECT_ROOT}/data/datasets"

#well_dataset_name = 'turbulent_radiative_layer_2D'
well_dataset_name = 'rayleigh_benard'
#well_dataset_name = 'planetswe'

stats_path = f"{well_base_path}/{well_dataset_name}/stats.yaml"

dataset = get_well_dataset(
    path=well_base_path,
    physics=well_dataset_name,
    split='test',
    steps=-1,
    transform = ['log_scalars'],   
    augment = [],
)

with fsspec.open(stats_path, mode="r") as f:
    stats = yaml.safe_load(f)

core_field_names          = dataset.core_field_names
core_constant_field_names = dataset.core_constant_field_names

normalizer = ZScoreNormalization(stats, core_field_names, core_constant_field_names)

# Load Baseline Models

In [ ]:
unet_classic  = UNetClassic.from_pretrained(f"polymathic-ai/UNetClassic-{well_dataset_name}")
unet_convnext = UNetConvNext.from_pretrained(f"polymathic-ai/UNetConvNext-{well_dataset_name}")
fno           = NeuralOperatorModule.from_pretrained(f"{PROJECT_ROOT}/experiments/{well_dataset_name}/ckpts/FNO_{well_dataset_name}_v1.0.pt")

baselines = {
    'UNetClassic'   : unet_classic.to(device),
    'UNetConvNext'  : unet_convnext.to(device),
    'FNO'           : fno.to(device),
}

In [9]:
N_TRAJ        = len(dataset.metadata.n_steps_per_trajectory)                # number of trajectories in the test set for this PDE 
N_STEPS_INPUT = 4                                                           # N_STEPS_INPUT sliding-window width (must match training)
ROLLOUT_STEPS = dataset.metadata.n_steps_per_trajectory[0] - N_STEPS_INPUT  # autoregressive steps after the seed window

baseline_results = {name: [] for name in baselines}
truth_list       = []   # one (T_total, H, W, C) per trajectory

for traj_idx in range(N_TRAJ):
    print(f"\n{'━'*60}")
    print(f"  Trajectory {traj_idx}")
    print(f"{'━'*60}")

    item = dataset[traj_idx]
    x_full, constants = get_well_inputs(item, device=device)   # (T_total, H, W, C)
    truth_list.append(x_full.cpu())

    # Run rollout
    for model_name, model in baselines.items():
        print(f"    Running rollout for {model_name}...")
        model.eval()
        y_pred_norm, y_ref_norm, y_pred_phys, y_ref_phys = rollout_normalized(
            model, x_full, normalizer,
            n_steps_input=N_STEPS_INPUT,
            rollout_steps=ROLLOUT_STEPS,
            device=device,
        )
        baseline_results[model_name].append(y_pred_phys)

# Stack to (N_TRAJ, N_ROLLOUT_STEPS, H, W, C)
for name in baselines:
    baseline_results[name] = torch.stack(baseline_results[name], dim=0)

truth_traj = torch.stack(truth_list, dim=0)   # (N_TRAJ, T_total, H, W, C)


━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  Trajectory 0
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
    Running rollout for UNetClassic...
    Running rollout for UNetConvNext...
    Running rollout for FNO...

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  Trajectory 1
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
    Running rollout for UNetClassic...
    Running rollout for UNetConvNext...
    Running rollout for FNO...

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  Trajectory 2
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
    Running rollout for UNetClassic...
    Running rollout for UNetConvNext...
    Running rollout for FNO...

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  Trajectory 3
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
    Running rollout for UNetClassic...
    Running rollout for UNetConvNext...
    Running rollout for FNO...

━━━━━━━

# Save baselines to binary

In [ ]:
# save predicted trajectories to disk
save_dir = Path(f'{PROJECT_ROOT}/experiments/{well_dataset_name}/results')
save_dir.mkdir(exist_ok=True)

for model_name in baselines.keys():
    save_path = save_dir / f"{model_name}_{well_dataset_name}_rollout_traj.pkl"
    with open(save_path, "wb") as f:
        pickle.dump(baseline_results[model_name], f)

# Save Ground-truth to binary

In [18]:
# save predicted trajectories to disk
save_dir = Path(f'{PROJECT_ROOT}/experiments/{well_dataset_name}/results')
save_dir.mkdir(exist_ok=True)

for model_name in baselines.keys():
    save_path = save_dir / f"ground_truth_{well_dataset_name}_traj.pkl"
    with open(save_path, "wb") as f:
        pickle.dump(truth_traj, f)